# Intelligent Product Recommendation - Model Training
This notebook trains both the Apriori and FP-Growth models to compare their performance (as required by the project), and then extracts high-confidence rules (e.g., > 50% confidence) for the backend.

In [1]:
!pip install mlxtend pandas scikit-learn joblib

In [8]:
import pandas as pd
import numpy as np
import time
from mlxtend.frequent_patterns import apriori, fpgrowth, association_rules
from mlxtend.preprocessing import TransactionEncoder
import joblib
import os

# Ensure models directory exists
os.makedirs('models', exist_ok=True)

# Load Dataset 
try:
    df = pd.read_csv(r'C:\Users\zamba\OneDrive\Desktop\TEMP\TNS\Intelligent Product Recommendation\data\transactions.csv', encoding='latin1')
except FileNotFoundError:
    print('Please place your dataset at /content/transactions.csv or data/transactions.csv')
    # For demonstration, creating a dummy dataset if not found
    df = pd.DataFrame({
        'InvoiceNo': [1, 1, 1, 2, 2, 3, 3, 3, 4, 4, 5, 5],
        'Description': ['Laptop', 'Mouse', 'Laptop Bag', 'Laptop', 'Keyboard', 'Laptop', 'Mouse', 'Keyboard', 'Mouse', 'Laptop Bag', 'Laptop', 'Mouse']
    })
df.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,12/1/2010 8:26,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,12/1/2010 8:26,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,12/1/2010 8:26,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,12/1/2010 8:26,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,12/1/2010 8:26,3.39,17850.0,United Kingdom


In [9]:
# Data Cleaning & Preprocessing
# Drop rows where Description or InvoiceNo is missing, and convert Description to string
cleaned_df = df.dropna(subset=['InvoiceNo', 'Description'])
cleaned_df = cleaned_df[cleaned_df['Description'].astype(str).str.strip() != '']

# Filter out cancelled orders (where InvoiceNo starts with 'C')
cleaned_df = cleaned_df[~cleaned_df['InvoiceNo'].astype(str).str.startswith('C')]

# Optional but recommended for the Kaggle dataset: Filter to a specific country to reduce size and speed up Apriori
if 'Country' in cleaned_df.columns:
    cleaned_df = cleaned_df[cleaned_df['Country'] == 'United Kingdom']

# Group transactions as lists of string products
transactions = cleaned_df.groupby('InvoiceNo')['Description'].apply(lambda x: [str(item).strip() for item in x]).tolist()

# One-Hot Encoding for mlxtend
te = TransactionEncoder()
te_ary = te.fit(transactions).transform(transactions)
df_encoded = pd.DataFrame(te_ary, columns=te.columns_)
print(f"Encoded Dataset Shape: {df_encoded.shape}")
df_encoded.head()

Encoded Dataset Shape: (18668, 4176)


,*Boombox Ipod Classic,*USB Office Mirror Ball,10 COLOUR SPACEBOY PEN,12 COLOURED PARTY BALLOONS,12 DAISY PEGS IN WOOD BOX,12 EGG HOUSE PAINTED WOOD,12 HANGING EGGS HAND PAINTED,12 IVORY ROSE PEG PLACE SETTINGS,12 MESSAGE CARDS WITH ENVELOPES,12 PENCIL SMALL TUBE WOODLAND,...,wrongly coded 20713,wrongly coded 23343,wrongly coded-23343,wrongly marked,wrongly marked 23343,wrongly marked carton 22804,wrongly marked. 23343 in box,wrongly sold (22719) barcode,wrongly sold as sets,wrongly sold sets
0,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
1,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
2,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
3,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
4,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False


In [10]:
# --- ALGORITHM COMPARISON ---
# We use a slightly higher min_support (0.02 or 0.03) so Apriori doesn't hang on massive datasets
min_support_threshold = 0.03

# 1. Apriori
print("Running Apriori...")
start_time = time.time()
apriori_itemsets = apriori(df_encoded, min_support=min_support_threshold, use_colnames=True)
apriori_time = time.time() - start_time

# 2. FP-Growth
print("Running FP-Growth...")
start_time = time.time()
fpgrowth_itemsets = fpgrowth(df_encoded, min_support=min_support_threshold, use_colnames=True)
fpgrowth_time = time.time() - start_time

print("\n--- Algorithm Performance ---")
print(f"Apriori Runtime: {apriori_time:.5f} seconds")
print(f"FP-Growth Runtime: {fpgrowth_time:.5f} seconds")
print(f"Itemsets Found (Apriori): {len(apriori_itemsets)}")
print(f"Itemsets Found (FP-Growth): {len(fpgrowth_itemsets)}")
print("\nNotice that FP-Growth is typically much faster for large datasets!")

Running Apriori...
Running FP-Growth...

--- Algorithm Performance ---
Apriori Runtime: 0.36130 seconds
FP-Growth Runtime: 1.49894 seconds
Itemsets Found (Apriori): 131
Itemsets Found (FP-Growth): 131

Notice that FP-Growth is typically much faster for large datasets!


In [11]:
# Generate Association Rules (using FP-Growth results since it's faster and yields the same itemsets)
rules = association_rules(fpgrowth_itemsets, metric='confidence', min_threshold=0.3)

# FILTERING FOR HIGH CONFIDENCE (50%+ Confidence)
highly_confident_rules = rules[rules['confidence'] >= 0.50]

print(f'\nGenerated {len(rules)} total rules.')
print(f'Generated {len(highly_confident_rules)} rules with >50% confidence.')
highly_confident_rules.head()


Generated 15 total rules.
Generated 10 rules with >50% confidence.


,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
1,frozenset({JUMBO BAG PINK POLKADOT}),frozenset({JUMBO BAG RED RETROSPOT}),0.062085,0.103814,0.042051,0.677308,6.524245,1.0,0.035605,2.777218,0.902774,0.339533,0.639927,0.541182
3,frozenset({JUMBO STORAGE BAG SUKI}),frozenset({JUMBO BAG RED RETROSPOT}),0.060531,0.103814,0.037390,0.617699,5.950055,1.0,0.031106,2.344190,0.885537,0.294515,0.573413,0.488932
4,frozenset({JUMBO BAG BAROQUE BLACK WHITE}),frozenset({JUMBO BAG RED RETROSPOT}),0.048747,0.103814,0.030534,0.626374,6.033613,1.0,0.025473,2.398615,0.877013,0.250219,0.583093,0.460246
6,frozenset({JUMBO SHOPPER VINTAGE RED PAISLEY}),frozenset({JUMBO BAG RED RETROSPOT}),0.060692,0.103814,0.035194,0.579876,5.585724,1.0,0.028893,2.133149,0.874018,0.272162,0.531209,0.459443
7,frozenset({ALARM CLOCK BAKELIKE RED}),frozenset({ALARM CLOCK BAKELIKE GREEN}),0.049818,0.046925,0.030159,0.605376,12.900874,1.0,0.027821,2.415149,0.970852,0.452936,0.585947,0.624035


In [12]:
# Save the artifacts for the backend to use
joblib.dump(fpgrowth_itemsets, 'models/frequent_itemsets.pkl')
joblib.dump(highly_confident_rules, 'models/association_rules.pkl')
# Also save product columns for reference
joblib.dump(list(df_encoded.columns), 'models/product_mapping.pkl')

print('Models saved successfully in models/ directory!')
print('You can now start your FastAPI backend.')

Models saved successfully in models/ directory!
You can now start your FastAPI backend.
